# 00 · Explorar el corpus

Qué hay en la carpeta de la empresa antes de construir ningún agente. Cada celda usa una pieza de la **capa de datos**
(`b5t4.corpus`), la misma que usarán los tres agentes.

Kernel: el entorno del repo (`.venv`). La ruta de datos sale de `.env` (`B5T4_DATA_DIR`).

In [ ]:
# 1 · Configuración: ¿a qué carpeta apunta el repo?
from b5t4.config import get_settings
s = get_settings()
print("Datos:", s.data_dir)
print("Manifiesto:", s.manifest_path.exists())

In [ ]:
# 2 · El manifiesto: una fila por fichero con su estado. Lo vigente es lo único que debería leer un agente.
import pandas as pd
from b5t4.corpus import load_manifest, summary

files = load_manifest()
df = pd.DataFrame([f.__dict__ for f in files]).drop(columns=["path"])
print(summary(files)["por_estado"])
df.pivot_table(index="folder", columns="status", values="rel", aggfunc="count", fill_value=0)

In [ ]:
# 3 · Los contextos: 20 documentos con la MISMA plantilla de 11 secciones.
from b5t4.corpus import load_contexts, section_index

docs = load_contexts()
alpha = next(d for d in docs if d.rel.startswith("10. Seccion Alpha"))
print([f"§{x.number} {x.title}" for x in alpha.sections])
print(alpha.section("2").text[:1500])   # §2 Responsables de la Sección Alpha

In [ ]:
# 4 · Vista transversal: la misma sección en todas las carpetas (lo que hace fácil el RAG y el guion del onboarding).
for sec in section_index(docs)["2"][:6]:
    print("─", sec.label, "·", len(sec.text.split()), "palabras")

In [ ]:
# 5 · Troceado para el RAG: cada fragmento lleva carpeta, sección, tipo y ruta (para filtrar y para citar).
from b5t4.agents.rag import build_chunks

chunks = build_chunks(include_documents=False)       # solo contextos (rápido); True = también documentos
hits = [c for c in chunks if "Luxemburgo" in c.text]
for c in hits[:5]:
    print(c.source, "·", c.heading)

In [ ]:
# 6 · Agente 2: ediciones de la Opinión Corporativa y texto de una página.
from b5t4.agents.opinion import list_editions
from b5t4.corpus import pdf_pages

eds = list_editions()
for e in eds:
    print(e.date, e.label)
print(pdf_pages(eds[-1].path)[2].text[:800])          # tendencias estructurales de la última edición